
# Day 34 — Personalization / Few-shot Adaptation

Notebook **standalone** cho hai dataset view đã hoàn thành ETL và baseline:

- `mendeley`: Mendeley Primary-4, 40 subjects, 5 repetitions/class.
- `grabmyo`: GRABMyo Primary-4 Forearm-16, 43 subjects, 3 sessions, 7 trials/class/session.

## Mục tiêu

So sánh theo cặp trên cùng evaluation trials:

- **P0 — Global frozen baseline:** dùng chính xác model Day 32, không cập nhật bằng dữ liệu người mới.
- **P1 — Few-shot personalized prototype adapter:** dùng `k=2` hoặc `k=3` labeled trials/class của subject mới để tạo prototype cá nhân; baseline và feature extractor vẫn frozen.

## Nguyên tắc chống leakage

1. Baseline Day 32 không được refit.
2. Adapter scaler và hyperparameter policy chỉ được học/chọn trên **training subjects**.
3. Validation subjects không tham gia policy selection.
4. Calibration trial và evaluation trial của cùng episode phải rời nhau hoàn toàn.
5. P0 và P1 được đánh giá trên đúng cùng evaluation IDs.
6. Với GRABMyo, calibration chỉ lấy từ Session 1; đánh giá riêng same-session và cross-session.
7. Không pool Mendeley và GRABMyo trong một lần chạy.
8. Không dùng kết quả này cho quyết định lâm sàng.

## Chọn dataset trước khi chạy

Trong một cell riêng trước khi Run all, có thể đặt:

```python
import os
os.environ["DAY34_DATASET_PROFILE"] = "mendeley"  # hoặc "grabmyo"
```

Mặc định là `mendeley`.


In [18]:
import os
os.environ["DAY34_DATASET_PROFILE"] = "grabmyo"


## Bước 1 — Môi trường, governance và cấu hình

**Input:** dataset profile và các artifact ETL/Day 32 trên Google Drive.  
**Output:** frozen run contract, runtime path và persistent path.  
**PASS:** chỉ một dataset profile được kích hoạt; test set và clinical use vẫn đóng.


In [19]:

# === CELL 1: Environment, governance, dataset profile and paths ===
from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import shutil
import time
import warnings
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

import joblib
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import scipy
from scipy.stats import wilcoxon

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.preprocessing import StandardScaler

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive, files  # type: ignore
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')

# -----------------------------------------------------------------------------
# Governance
# -----------------------------------------------------------------------------
DAY34_PERSONALIZATION_AUTHORIZED = True
BASELINE_REFIT_ALLOWED = False
VALIDATION_POLICY_TUNING_ALLOWED = False
TEST_SET_OPENED = False
POOLED_DATASET_ALLOWED = False
CLINICAL_USE_ALLOWED = False

assert DAY34_PERSONALIZATION_AUTHORIZED is True
assert BASELINE_REFIT_ALLOWED is False
assert VALIDATION_POLICY_TUNING_ALLOWED is False
assert TEST_SET_OPENED is False
assert POOLED_DATASET_ALLOWED is False
assert CLINICAL_USE_ALLOWED is False

# -----------------------------------------------------------------------------
# Run configuration
# -----------------------------------------------------------------------------
DATASET_PROFILE = os.environ.get('DAY34_DATASET_PROFILE', 'mendeley').strip().lower()
if DATASET_PROFILE not in {'mendeley', 'grabmyo'}:
    raise ValueError("DAY34_DATASET_PROFILE phải là 'mendeley' hoặc 'grabmyo'.")

RUN_ID = f'day34-personalization-{DATASET_PROFILE}-v2'
K_VALUES = (2, 3)
SEEDS = tuple(range(3401, 3421))
BOOTSTRAP_RESAMPLES = 5000
RANDOM_SEED = 3401
CHUNK_SIZE = 25000
FORCE_RECOMPUTE_POLICY = False
DOWNLOAD_HANDOFF_TO_BROWSER = False
QUICK_SMOKE_TEST = os.environ.get('DAY34_QUICK_SMOKE', '0') == '1'

# Small, frozen policy grid. Selection happens only on training subjects.
POLICY_GRID = [
    {
        'distance_metric': metric,
        'subject_prototype_weight': rho,
        'personalization_blend_weight': alpha,
        'temperature': temperature,
    }
    for metric in ('cosine', 'sqeuclidean')
    for rho in (0.50, 0.75, 1.00)
    for alpha in (0.25, 0.50, 0.75)
    for temperature in (0.50, 1.00)
]

if QUICK_SMOKE_TEST:
    SEEDS = SEEDS[:2]
    POLICY_GRID = POLICY_GRID[:4]
    BOOTSTRAP_RESAMPLES = 200

DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/MyoLab-AI-data')

PROFILE_CONFIG = {
    'mendeley': {
        'dataset_slug': 'mendeley-4channel-hand-gesture-v2',
        'npz_relative': Path('outputs/day31-mendeley-primary-fall14.npz'),
        'baseline_relative': Path('outputs/day32-baseline/day32-baseline-v1'),
        'model_name': 'day32-selected-model.joblib',
        'metadata_name': 'day32-selected-model-metadata.json',
        'gate_name': 'day32-final-gate.json',
        'baseline_predictions_name': 'day32-validation-repetition-predictions.csv',
        'baseline_prediction_key': 'repetition_id',
        'trial_key_source': 'repetition_ids',
        'calibration_pool': 'all_trials',
        'primary_selection_scope': 'all_eval',
        'expected_subjects': 40,
        'expected_train_subjects': 32,
        'expected_validation_subjects': 8,
        'expected_classes': 4,
        'expected_trials_per_class': 5,
        'p0_aggregation': 'window_vote_fraction',
    },
    'grabmyo': {
        'dataset_slug': 'grabmyo-physionet-v1.1.0',
        'npz_relative': Path('outputs/grabmyo-primary4-forearm16-fall14.npz'),
        'baseline_relative': Path('outputs/baseline/grabmyo-primary4-baseline-v1'),
        'model_name': 'grabmyo-selected-model.joblib',
        'metadata_name': 'grabmyo-selected-model-metadata.json',
        'gate_name': 'grabmyo-baseline-final-gate.json',
        'baseline_predictions_name': 'grabmyo-validation-trial-predictions.csv',
        'baseline_prediction_key': 'record_id',
        'trial_key_source': 'record_id',
        'calibration_pool': 'session1_only',
        'primary_selection_scope': 'cross_session',
        'expected_subjects': 43,
        'expected_train_subjects': 34,
        'expected_validation_subjects': 9,
        'expected_classes': 4,
        'expected_trials_per_class': 21,
        'p0_aggregation': 'mean_window_model_score',
    },
}
PROFILE = PROFILE_CONFIG[DATASET_PROFILE]
DATASET_ROOT = DRIVE_DATA_ROOT / PROFILE['dataset_slug']

if IN_COLAB:
    RUNTIME_ROOT = Path('/content/data') / PROFILE['dataset_slug'] / 'day34-personalization' / RUN_ID
else:
    RUNTIME_ROOT = Path.cwd() / '.day34-runtime' / PROFILE['dataset_slug'] / RUN_ID

PERSIST_ROOT = DATASET_ROOT / 'outputs' / 'day34-personalization' / RUN_ID
RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
PERSIST_ROOT.mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_SEED)

print('=' * 92)
print('[CELL 1] DAY 34 CONFIGURATION')
print('=' * 92)
print(f'Dataset profile               : {DATASET_PROFILE}')
print(f'Dataset root                  : {DATASET_ROOT}')
print(f'Runtime output                : {RUNTIME_ROOT}')
print(f'Persistent output             : {PERSIST_ROOT}')
print(f'k values                      : {K_VALUES}')
print(f'Episode seeds                 : {len(SEEDS)} ({SEEDS[0]}..{SEEDS[-1]})')
print(f'Policy candidates             : {len(POLICY_GRID)}')
print(f'Calibration pool              : {PROFILE["calibration_pool"]}')
print(f'Primary selection scope       : {PROFILE["primary_selection_scope"]}')
print(f'Quick smoke test              : {QUICK_SMOKE_TEST}')
print(f'Baseline refit allowed        : {BASELINE_REFIT_ALLOWED}')
print(f'Validation policy tuning      : {VALIDATION_POLICY_TUNING_ALLOWED}')
print(f'Test set opened               : {TEST_SET_OPENED}')
print(f'Clinical use allowed          : {CLINICAL_USE_ALLOWED}')
print('[PASS] Governance and configuration initialized.')


[CELL 1] DAY 34 CONFIGURATION
Dataset profile               : grabmyo
Dataset root                  : /content/drive/MyDrive/MyoLab-AI-data/grabmyo-physionet-v1.1.0
Runtime output                : /content/data/grabmyo-physionet-v1.1.0/day34-personalization/day34-personalization-grabmyo-v2
Persistent output             : /content/drive/MyDrive/MyoLab-AI-data/grabmyo-physionet-v1.1.0/outputs/day34-personalization/day34-personalization-grabmyo-v2
k values                      : (2, 3)
Episode seeds                 : 20 (3401..3420)
Policy candidates             : 36
Calibration pool              : session1_only
Primary selection scope       : cross_session
Quick smoke test              : False
Baseline refit allowed        : False
Validation policy tuning      : False
Test set opened               : False
Clinical use allowed          : False
[PASS] Governance and configuration initialized.



## Bước 2 — Helpers dùng chung

**Input:** không có dữ liệu.  
**Output:** hàm hash, atomic write, score alignment, metrics, bootstrap và deterministic episode sampling.  
**PASS:** toàn bộ helper được định nghĩa, không phụ thuộc module nội bộ trong repository.


In [20]:

# === CELL 2: Common helpers ===
def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as file_obj:
        while True:
            chunk = file_obj.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe(item) for item in value]
    if isinstance(value, np.ndarray):
        return json_safe(value.tolist())
    if isinstance(value, np.generic):
        return json_safe(value.item())
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def write_json_atomic(payload: dict[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.part')
    temporary.write_text(
        json.dumps(json_safe(payload), ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
    temporary.replace(path)


def write_dataframe_atomic(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.part')
    compression = 'gzip' if path.suffix == '.gz' else None
    frame.to_csv(temporary, index=False, compression=compression)
    temporary.replace(path)


def persist_artifact(runtime_path: Path) -> Path:
    persistent_path = PERSIST_ROOT / runtime_path.name
    temporary = persistent_path.with_suffix(persistent_path.suffix + '.part')
    shutil.copy2(runtime_path, temporary)
    temporary.replace(persistent_path)
    return persistent_path


def resolve_artifact(
    *,
    override_env: str,
    exact_path: Path,
    search_root: Path,
    filename: str,
) -> Path:
    override = os.environ.get(override_env, '').strip()
    candidates: list[Path] = []
    if override:
        candidates.append(Path(override))
    candidates.append(exact_path)
    for candidate in candidates:
        if candidate.exists():
            return candidate
    matches = sorted(search_root.rglob(filename)) if search_root.exists() else []
    if len(matches) == 1:
        return matches[0]
    if not matches:
        raise FileNotFoundError(
            f'Không tìm thấy {filename}. Exact path: {exact_path}. '
            f'Có thể đặt biến môi trường {override_env}.'
        )
    raise RuntimeError(
        f'Tìm thấy nhiều artifact tên {filename}; cần đặt {override_env}:\n'
        + '\n'.join(str(path) for path in matches)
    )


def scalar_text(array: np.ndarray) -> str:
    return str(np.asarray(array).reshape(-1)[0])


def stable_seed(*parts: Any) -> int:
    text = '||'.join(str(part) for part in parts)
    digest = hashlib.sha256(text.encode('utf-8')).digest()
    return int.from_bytes(digest[:8], 'little') % (2**32 - 1)


def estimator_classes(estimator: Any) -> np.ndarray:
    if hasattr(estimator, 'classes_'):
        return np.asarray(estimator.classes_).astype(str)
    if hasattr(estimator, 'named_steps'):
        for step in reversed(list(estimator.named_steps.values())):
            if hasattr(step, 'classes_'):
                return np.asarray(step.classes_).astype(str)
    raise RuntimeError('Không tìm thấy classes_ trong frozen baseline model.')


def aligned_score_matrix(
    estimator: Any,
    X_input: np.ndarray,
    class_order: np.ndarray,
) -> tuple[np.ndarray, str]:
    if hasattr(estimator, 'predict_proba'):
        raw = np.asarray(estimator.predict_proba(X_input), dtype=np.float64)
        score_kind = 'predict_proba'
    elif hasattr(estimator, 'decision_function'):
        raw = np.asarray(estimator.decision_function(X_input), dtype=np.float64)
        if raw.ndim == 1:
            raw = np.column_stack([-raw, raw])
        score_kind = 'decision_function'
    else:
        predictions = np.asarray(estimator.predict(X_input)).astype(str)
        raw = np.zeros((len(predictions), len(class_order)), dtype=np.float64)
        for class_index, label in enumerate(class_order):
            raw[:, class_index] = predictions == label
        return raw, 'one_hot_prediction'

    classes = estimator_classes(estimator)
    aligned = np.full((raw.shape[0], len(class_order)), np.nan, dtype=np.float64)
    for source_index, label in enumerate(classes):
        matches = np.flatnonzero(class_order == label)
        if len(matches) != 1:
            raise RuntimeError(f'Estimator class không thuộc frozen class order: {label}')
        aligned[:, int(matches[0])] = raw[:, source_index]
    if not np.isfinite(aligned).all():
        raise RuntimeError('Không align được toàn bộ score columns với class_order.')
    return aligned, score_kind


def softmax_rows(values: np.ndarray, temperature: float = 1.0) -> np.ndarray:
    if temperature <= 0:
        raise ValueError('temperature phải > 0.')
    shifted = values / float(temperature)
    shifted = shifted - np.max(shifted, axis=1, keepdims=True)
    exp_values = np.exp(shifted)
    denominator = exp_values.sum(axis=1, keepdims=True)
    if np.any(denominator <= 0) or not np.isfinite(denominator).all():
        raise RuntimeError('Softmax denominator không hợp lệ.')
    return exp_values / denominator


def normalize_global_trial_scores(values: np.ndarray, score_kind: str) -> np.ndarray:
    values = np.asarray(values, dtype=np.float64)
    if score_kind in {'predict_proba', 'one_hot_prediction', 'window_vote_fraction'}:
        clipped = np.clip(values, 0.0, None)
        denominator = clipped.sum(axis=1, keepdims=True)
        if np.any(denominator <= 0):
            raise RuntimeError('Global probability row có tổng bằng 0.')
        return clipped / denominator
    return softmax_rows(values, temperature=1.0)


def classification_metrics(y_true: Iterable[str], y_pred: Iterable[str], labels: np.ndarray) -> dict[str, float]:
    y_true = np.asarray(list(y_true)).astype(str)
    y_pred = np.asarray(list(y_pred)).astype(str)
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'macro_f1': float(f1_score(y_true, y_pred, labels=labels, average='macro', zero_division=0)),
        'macro_precision': float(precision_score(y_true, y_pred, labels=labels, average='macro', zero_division=0)),
        'macro_recall': float(recall_score(y_true, y_pred, labels=labels, average='macro', zero_division=0)),
    }


def score_diagnostics(probabilities: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    ordered = np.sort(probabilities, axis=1)
    confidence = ordered[:, -1]
    margin = ordered[:, -1] - ordered[:, -2]
    entropy = -np.sum(probabilities * np.log(np.clip(probabilities, 1e-12, 1.0)), axis=1)
    return confidence, margin, entropy


def bootstrap_mean_ci(subject_values: np.ndarray, seed: int, n_resamples: int) -> tuple[float, float]:
    values = np.asarray(subject_values, dtype=np.float64)
    if len(values) == 0:
        return float('nan'), float('nan')
    rng = np.random.default_rng(seed)
    sample_indices = rng.integers(0, len(values), size=(n_resamples, len(values)))
    means = values[sample_indices].mean(axis=1)
    return float(np.quantile(means, 0.025)), float(np.quantile(means, 0.975))


def metric_value(frame: pd.DataFrame, truth_col: str, pred_col: str, labels: np.ndarray) -> dict[str, float]:
    return classification_metrics(frame[truth_col].astype(str), frame[pred_col].astype(str), labels)


print('[PASS] Common helpers initialized.')


[PASS] Common helpers initialized.



## Bước 3 — Locate, load và audit ETL + frozen baseline

**Input:** canonical NPZ, selected model `joblib`, model metadata, Day 32 final gate và Day 32 validation predictions.  
**Output:** feature matrix, metadata, frozen model, exact feature arm và train/validation subject sets.  
**PASS:** checksum khớp, class order khớp, feature indices hợp lệ, subject split rời nhau và baseline gate đã PASS.


In [21]:

# === CELL 3: Locate and validate Day 31/GRABMyo ETL + Day 32 baseline artifacts ===
NPZ_PATH = resolve_artifact(
    override_env='DAY34_NPZ_PATH',
    exact_path=DATASET_ROOT / PROFILE['npz_relative'],
    search_root=DATASET_ROOT,
    filename=PROFILE['npz_relative'].name,
)
BASELINE_ROOT = DATASET_ROOT / PROFILE['baseline_relative']
MODEL_PATH = resolve_artifact(
    override_env='DAY34_BASELINE_MODEL_PATH',
    exact_path=BASELINE_ROOT / PROFILE['model_name'],
    search_root=DATASET_ROOT,
    filename=PROFILE['model_name'],
)
MODEL_METADATA_PATH = resolve_artifact(
    override_env='DAY34_BASELINE_METADATA_PATH',
    exact_path=BASELINE_ROOT / PROFILE['metadata_name'],
    search_root=DATASET_ROOT,
    filename=PROFILE['metadata_name'],
)
BASELINE_GATE_PATH = resolve_artifact(
    override_env='DAY34_BASELINE_GATE_PATH',
    exact_path=BASELINE_ROOT / PROFILE['gate_name'],
    search_root=DATASET_ROOT,
    filename=PROFILE['gate_name'],
)
BASELINE_PREDICTIONS_PATH = resolve_artifact(
    override_env='DAY34_BASELINE_PREDICTIONS_PATH',
    exact_path=BASELINE_ROOT / PROFILE['baseline_predictions_name'],
    search_root=DATASET_ROOT,
    filename=PROFILE['baseline_predictions_name'],
)

NPZ_SHA256 = sha256_file(NPZ_PATH)
MODEL_SHA256 = sha256_file(MODEL_PATH)
model_metadata = json.loads(MODEL_METADATA_PATH.read_text(encoding='utf-8'))
baseline_gate = json.loads(BASELINE_GATE_PATH.read_text(encoding='utf-8'))

if not str(baseline_gate.get('status', '')).startswith('PASS'):
    raise RuntimeError(f'Day 32 baseline gate chưa PASS: {baseline_gate.get("status")}')

with np.load(NPZ_PATH, allow_pickle=False) as data:
    common_required = {
        'X', 'y', 'subject_id', 'window_id', 'split_names',
        'feature_names', 'class_order', 'dataset_view_id',
    }
    profile_required = {'repetition_ids', 'record_id'} if DATASET_PROFILE == 'mendeley' else {
        'record_id', 'repetition_id', 'session_id', 'trial_id'
    }
    missing = (common_required | profile_required) - set(data.files)
    if missing:
        raise RuntimeError(f'Canonical NPZ thiếu keys: {sorted(missing)}')

    X = np.asarray(data['X'], dtype=np.float32)
    y = np.asarray(data['y']).astype(str)
    subject_ids = np.asarray(data['subject_id']).astype(str)
    window_ids = np.asarray(data['window_id']).astype(str)
    split_names = np.asarray(data['split_names']).astype(str)
    feature_names = np.asarray(data['feature_names']).astype(str)
    class_order = np.asarray(data['class_order']).astype(str)
    dataset_view_id = scalar_text(data['dataset_view_id'])

    if DATASET_PROFILE == 'mendeley':
        trial_keys = np.asarray(data['repetition_ids']).astype(str)
        record_ids = np.asarray(data['record_id']).astype(str)
        repetition_ids = trial_keys.copy()
        session_ids = np.zeros(len(y), dtype=np.int16)
        trial_ordinals = np.full(len(y), -1, dtype=np.int16)
    else:
        trial_keys = np.asarray(data['record_id']).astype(str)
        record_ids = trial_keys.copy()
        repetition_ids = np.asarray(data['repetition_id']).astype(str)
        session_ids = np.asarray(data['session_id']).astype(np.int16)
        trial_ordinals = np.asarray(data['trial_id']).astype(np.int16)

if X.ndim != 2:
    raise RuntimeError(f'X phải là 2D, nhận {X.shape}.')
if not np.isfinite(X).all():
    raise RuntimeError('Canonical X chứa NaN/Inf.')
if len({len(X), len(y), len(subject_ids), len(window_ids), len(split_names), len(trial_keys)}) != 1:
    raise RuntimeError('Các array trong NPZ không cùng số dòng.')
if len(np.unique(window_ids)) != len(window_ids):
    raise RuntimeError('NPZ chứa duplicate window_id.')
if len(np.unique(class_order)) != PROFILE['expected_classes']:
    raise RuntimeError(f'Primary view phải có {PROFILE["expected_classes"]} classes: {class_order.tolist()}')
if set(np.unique(y)) != set(class_order):
    raise RuntimeError('Observed labels không khớp frozen class_order.')

feature_indices = np.asarray(model_metadata.get('feature_indices', []), dtype=np.int64)
metadata_feature_names = np.asarray(model_metadata.get('feature_names', [])).astype(str)
metadata_class_order = np.asarray(model_metadata.get('class_order', [])).astype(str)

if feature_indices.ndim != 1 or len(feature_indices) == 0:
    raise RuntimeError('Model metadata thiếu feature_indices hợp lệ.')
if feature_indices.min() < 0 or feature_indices.max() >= X.shape[1]:
    raise RuntimeError('Model feature_indices vượt ngoài X.')
if not np.array_equal(feature_names[feature_indices], metadata_feature_names):
    raise RuntimeError('Feature names của NPZ không khớp Day 32 model metadata.')
if not np.array_equal(class_order, metadata_class_order):
    raise RuntimeError('Class order của NPZ không khớp Day 32 model metadata.')

metadata_npz_hash = model_metadata.get('matrix_sha256') or model_metadata.get('npz_sha256')
if metadata_npz_hash and metadata_npz_hash != NPZ_SHA256:
    raise RuntimeError('NPZ checksum không khớp model metadata.')
metadata_model_hash = model_metadata.get('model_sha256')
if metadata_model_hash and metadata_model_hash != MODEL_SHA256:
    raise RuntimeError('Model checksum không khớp model metadata.')

frozen_model = joblib.load(MODEL_PATH)
model_classes = estimator_classes(frozen_model)
if set(model_classes) != set(class_order):
    raise RuntimeError('Frozen model classes không khớp class_order.')

train_mask = split_names == 'train'
validation_mask = split_names == 'validation'
if np.any(~(train_mask | validation_mask)):
    raise RuntimeError(f'Unexpected split names: {np.unique(split_names).tolist()}')

train_subjects = sorted(set(subject_ids[train_mask].tolist()))
validation_subjects = sorted(set(subject_ids[validation_mask].tolist()))
all_subjects = sorted(set(subject_ids.tolist()))

if set(train_subjects) & set(validation_subjects):
    raise RuntimeError('Subject leakage giữa train và validation.')
if len(all_subjects) != PROFILE['expected_subjects']:
    raise RuntimeError(f'Expected {PROFILE["expected_subjects"]} subjects, got {len(all_subjects)}.')
if len(train_subjects) != PROFILE['expected_train_subjects']:
    raise RuntimeError(f'Expected {PROFILE["expected_train_subjects"]} train subjects, got {len(train_subjects)}.')
if len(validation_subjects) != PROFILE['expected_validation_subjects']:
    raise RuntimeError(f'Expected {PROFILE["expected_validation_subjects"]} validation subjects, got {len(validation_subjects)}.')

trial_contract = pd.DataFrame({
    'trial_key': trial_keys,
    'subject_id': subject_ids,
    'session_id': session_ids,
    'split_name': split_names,
    'y_true': y,
}).drop_duplicates()

if trial_contract.groupby('trial_key')['subject_id'].nunique().max() != 1:
    raise RuntimeError('Một trial_key map sang nhiều subjects.')
if trial_contract.groupby('trial_key')['y_true'].nunique().max() != 1:
    raise RuntimeError('Một trial_key map sang nhiều labels.')
if trial_contract.groupby('trial_key')['split_name'].nunique().max() != 1:
    raise RuntimeError('Một trial_key nằm trong nhiều partitions.')

INPUT_GATE_JSON = RUNTIME_ROOT / 'day34-input-gate.json'
input_gate = {
    'schema_version': 'day34-input-gate.v2',
    'created_at_utc': utc_now_iso(),
    'status': 'PASS',
    'dataset_profile': DATASET_PROFILE,
    'dataset_view_id': dataset_view_id,
    'npz_path': str(NPZ_PATH),
    'npz_sha256': NPZ_SHA256,
    'npz_shape': list(X.shape),
    'model_path': str(MODEL_PATH),
    'model_sha256': MODEL_SHA256,
    'model_metadata_path': str(MODEL_METADATA_PATH),
    'baseline_gate_path': str(BASELINE_GATE_PATH),
    'baseline_predictions_path': str(BASELINE_PREDICTIONS_PATH),
    'feature_arm': model_metadata.get('feature_arm'),
    'feature_dimension': int(len(feature_indices)),
    'class_order': class_order.tolist(),
    'train_subject_ids': train_subjects,
    'validation_subject_ids': validation_subjects,
    'checks': {
        'baseline_gate_pass': True,
        'npz_checksum_matches_metadata': bool(not metadata_npz_hash or metadata_npz_hash == NPZ_SHA256),
        'model_checksum_matches_metadata_when_available': bool(not metadata_model_hash or metadata_model_hash == MODEL_SHA256),
        'feature_contract_matches': True,
        'class_contract_matches': True,
        'subject_split_disjoint': True,
        'pooled_dataset': False,
    },
}
write_json_atomic(input_gate, INPUT_GATE_JSON)
persist_artifact(INPUT_GATE_JSON)

print('=' * 92)
print('[CELL 3] INPUT GATE')
print('=' * 92)
print(f'NPZ                          : {NPZ_PATH}')
print(f'NPZ SHA-256                  : {NPZ_SHA256}')
print(f'X shape                      : {X.shape}')
print(f'Dataset view                 : {dataset_view_id}')
print(f'Frozen model                 : {MODEL_PATH}')
print(f'Model SHA-256                : {MODEL_SHA256}')
print(f'Selected feature arm         : {model_metadata.get("feature_arm")}')
print(f'Selected feature dimension   : {len(feature_indices)}')
print(f'Classes                      : {class_order.tolist()}')
print(f'Train subjects               : {len(train_subjects)}')
print(f'Validation subjects          : {len(validation_subjects)}')
print('[PASS] ETL and frozen baseline artifacts validated.')


[CELL 3] INPUT GATE
NPZ                          : /content/drive/MyDrive/MyoLab-AI-data/grabmyo-physionet-v1.1.0/outputs/grabmyo-primary4-forearm16-fall14.npz
NPZ SHA-256                  : cbd4e0f7203d85c7acb81ea985b436675eb451022fcd11bacb6f3d0433f5a197
X shape                      : (173376, 224)
Dataset view                 : grabmyo-primary4-forearm16-fall14-v1
Frozen model                 : /content/drive/MyDrive/MyoLab-AI-data/grabmyo-physionet-v1.1.0/outputs/baseline/grabmyo-primary4-baseline-v1/grabmyo-selected-model.joblib
Model SHA-256                : cf4b2b254b53e7dfbcb7af03bfe2e8956531c8115424917b0dca2e60a04d8619
Selected feature arm         : F-ALL14
Selected feature dimension   : 224
Classes                      : ['rest', 'hand_close', 'wrist_flexion', 'wrist_extension']
Train subjects               : 34
Validation subjects          : 9
[PASS] ETL and frozen baseline artifacts validated.



## Bước 4 — Reproduce P0 và xây trial-level table

**Input:** frozen Day 32 model và selected feature arm.  
**Output:** một dòng cho mỗi repetition/trial với P0 score vector, P0 prediction và window coverage.  
**PASS:** P0 validation prediction khớp 100% artifact Day 32 đã bàn giao.


In [22]:

# === CELL 4: Reproduce exact P0 and aggregate windows to trial/repetition level ===
X_selected = X[:, feature_indices]
window_predictions = np.asarray(frozen_model.predict(X_selected)).astype(str)
window_scores, window_score_kind = aligned_score_matrix(frozen_model, X_selected, class_order)

window_frame = pd.DataFrame({
    'row_index': np.arange(len(y), dtype=np.int64),
    'trial_key': trial_keys,
    'record_id': record_ids,
    'repetition_id': repetition_ids,
    'subject_id': subject_ids,
    'session_id': session_ids,
    'trial_ordinal': trial_ordinals,
    'split_name': split_names,
    'y_true': y,
    'window_id': window_ids,
    'window_pred': window_predictions,
})

score_columns = [f'window_score__{label}' for label in class_order]
for class_index, column in enumerate(score_columns):
    window_frame[column] = window_scores[:, class_index]

trial_rows: list[dict[str, Any]] = []
for trial_key, group in window_frame.groupby('trial_key', sort=False):
    unique_checks = {
        'subject_id': group['subject_id'].unique(),
        'session_id': group['session_id'].unique(),
        'split_name': group['split_name'].unique(),
        'y_true': group['y_true'].unique(),
        'record_id': group['record_id'].unique(),
        'repetition_id': group['repetition_id'].unique(),
    }
    if any(len(values) != 1 for values in unique_checks.values()):
        raise RuntimeError(f'Inconsistent metadata trong trial {trial_key}: {unique_checks}')

    if PROFILE['p0_aggregation'] == 'window_vote_fraction':
        counts = np.asarray([(group['window_pred'] == label).sum() for label in class_order], dtype=np.float64)
        raw_trial_scores = counts / counts.sum()
        trial_score_kind = 'window_vote_fraction'
    else:
        raw_trial_scores = group[score_columns].to_numpy(dtype=np.float64).mean(axis=0)
        trial_score_kind = window_score_kind

    p0_pred = str(class_order[int(np.argmax(raw_trial_scores))])
    row = {
        'trial_key': str(trial_key),
        'record_id': str(unique_checks['record_id'][0]),
        'repetition_id': str(unique_checks['repetition_id'][0]),
        'subject_id': str(unique_checks['subject_id'][0]),
        'session_id': int(unique_checks['session_id'][0]),
        'trial_ordinal': int(group['trial_ordinal'].iloc[0]),
        'split_name': str(unique_checks['split_name'][0]),
        'y_true': str(unique_checks['y_true'][0]),
        'valid_window_count': int(len(group)),
        'p0_pred': p0_pred,
        'p0_correct': bool(p0_pred == str(unique_checks['y_true'][0])),
        'global_score_kind': trial_score_kind,
    }
    for class_index, label in enumerate(class_order):
        row[f'global_raw__{label}'] = float(raw_trial_scores[class_index])
    trial_rows.append(row)

trial_df = pd.DataFrame(trial_rows).sort_values(
    ['split_name', 'subject_id', 'session_id', 'trial_key']
).reset_index(drop=True)

raw_global_matrix = trial_df[[f'global_raw__{label}' for label in class_order]].to_numpy(dtype=np.float64)
global_prob_matrix = normalize_global_trial_scores(raw_global_matrix, trial_df['global_score_kind'].iloc[0])
for class_index, label in enumerate(class_order):
    trial_df[f'global_prob__{label}'] = global_prob_matrix[:, class_index]

p0_confidence, p0_margin, p0_entropy = score_diagnostics(global_prob_matrix)
trial_df['p0_confidence'] = p0_confidence
trial_df['p0_margin'] = p0_margin
trial_df['p0_entropy'] = p0_entropy

# Exact Day 32 reproduction audit on frozen validation trials.
baseline_predictions_df = pd.read_csv(BASELINE_PREDICTIONS_PATH, dtype=str)
baseline_key = PROFILE['baseline_prediction_key']
if baseline_key not in baseline_predictions_df.columns or 'y_pred' not in baseline_predictions_df.columns:
    raise RuntimeError(
        f'Baseline predictions thiếu {baseline_key} hoặc y_pred: '
        f'{baseline_predictions_df.columns.tolist()}'
    )

baseline_reference = baseline_predictions_df[[baseline_key, 'y_pred']].drop_duplicates().rename(
    columns={baseline_key: 'trial_key', 'y_pred': 'day32_y_pred'}
)
validation_p0 = trial_df[trial_df['split_name'] == 'validation'][
    ['trial_key', 'subject_id', 'session_id', 'y_true', 'p0_pred']
].merge(baseline_reference, on='trial_key', how='left', validate='one_to_one')

if validation_p0['day32_y_pred'].isna().any():
    missing_keys = validation_p0.loc[validation_p0['day32_y_pred'].isna(), 'trial_key'].tolist()
    raise RuntimeError(f'Day 32 artifact thiếu validation trial IDs: {missing_keys[:10]}')

validation_p0['match_day32'] = validation_p0['p0_pred'] == validation_p0['day32_y_pred']
p0_mismatch_count = int((~validation_p0['match_day32']).sum())
if p0_mismatch_count != 0:
    display(validation_p0[~validation_p0['match_day32']].head(20))
    raise RuntimeError(f'P0 reproduction mismatch với Day 32: {p0_mismatch_count} trials.')

P0_AUDIT_CSV = RUNTIME_ROOT / 'day34-p0-reproduction-audit.csv'
P0_AUDIT_JSON = RUNTIME_ROOT / 'day34-p0-reproduction-audit.json'
write_dataframe_atomic(validation_p0, P0_AUDIT_CSV)
write_json_atomic({
    'schema_version': 'day34-p0-reproduction.v1',
    'created_at_utc': utc_now_iso(),
    'status': 'PASS',
    'dataset_profile': DATASET_PROFILE,
    'validation_trial_count': int(len(validation_p0)),
    'mismatch_count': p0_mismatch_count,
    'p0_aggregation': PROFILE['p0_aggregation'],
    'window_score_kind': window_score_kind,
}, P0_AUDIT_JSON)
for path in (P0_AUDIT_CSV, P0_AUDIT_JSON):
    persist_artifact(path)

print('=' * 92)
print('[CELL 4] P0 REPRODUCTION')
print('=' * 92)
print(f'Window rows                   : {len(window_frame)}')
print(f'Trial/repetition rows         : {len(trial_df)}')
print(f'P0 aggregation                : {PROFILE["p0_aggregation"]}')
print(f'Window score kind             : {window_score_kind}')
print(f'Validation trials audited     : {len(validation_p0)}')
print(f'Day 32 mismatches             : {p0_mismatch_count}')
print('[PASS] Frozen P0 reproduced exactly.')


[CELL 4] P0 REPRODUCTION
Window rows                   : 173376
Trial/repetition rows         : 3612
P0 aggregation                : mean_window_model_score
Window score kind             : decision_function
Validation trials audited     : 756
Day 32 mismatches             : 0
[PASS] Frozen P0 reproduced exactly.



## Bước 5 — Frozen adapter representation

**Input:** selected Day 32 features.  
**Output:** `StandardScaler` fit chỉ trên training windows, trial centroids và global class prototypes.  
**PASS:** scaler không thấy validation rows; mỗi trial có đúng một centroid hữu hạn.


In [23]:

# === CELL 5: Fit train-only adapter scaler and build trial embeddings ===
adapter_scaler = StandardScaler(copy=True)
adapter_scaler.fit(X_selected[train_mask])

# Transform in chunks to avoid a second large full-precision matrix in RAM.
trial_position_map = {trial_key: index for index, trial_key in enumerate(trial_df['trial_key'].astype(str))}
window_trial_positions = pd.Series(trial_keys).map(trial_position_map).to_numpy()
if np.any(pd.isna(window_trial_positions)):
    raise RuntimeError('Không map được window rows sang trial table.')
window_trial_positions = window_trial_positions.astype(np.int64)

embedding_dim = len(feature_indices)
trial_embedding_sum = np.zeros((len(trial_df), embedding_dim), dtype=np.float64)
trial_embedding_count = np.zeros(len(trial_df), dtype=np.int64)

for start in range(0, len(X_selected), CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, len(X_selected))
    transformed = adapter_scaler.transform(X_selected[start:end]).astype(np.float32, copy=False)
    positions = window_trial_positions[start:end]
    np.add.at(trial_embedding_sum, positions, transformed)
    np.add.at(trial_embedding_count, positions, 1)

if np.any(trial_embedding_count <= 0):
    raise RuntimeError('Có trial không có window để tạo embedding.')
trial_embeddings = trial_embedding_sum / trial_embedding_count[:, None]
if not np.isfinite(trial_embeddings).all():
    raise RuntimeError('Trial embeddings chứa NaN/Inf.')

# Equal weight per trial when building global prototypes.
train_trial_mask = trial_df['split_name'].to_numpy() == 'train'
global_class_prototypes = np.zeros((len(class_order), embedding_dim), dtype=np.float64)
for class_index, label in enumerate(class_order):
    class_rows = train_trial_mask & (trial_df['y_true'].to_numpy() == label)
    if not np.any(class_rows):
        raise RuntimeError(f'Training trials thiếu class {label}.')
    global_class_prototypes[class_index] = trial_embeddings[class_rows].mean(axis=0)

ADAPTER_REPRESENTATION_JSON = RUNTIME_ROOT / 'day34-adapter-representation.json'
write_json_atomic({
    'schema_version': 'day34-adapter-representation.v1',
    'created_at_utc': utc_now_iso(),
    'status': 'PASS',
    'scaler_fit_partition': 'train_only',
    'training_window_count': int(train_mask.sum()),
    'validation_window_count_seen_by_scaler': 0,
    'trial_count': int(len(trial_df)),
    'embedding_dimension': int(embedding_dim),
    'global_prototype_shape': list(global_class_prototypes.shape),
    'feature_indices': feature_indices.tolist(),
    'feature_names': feature_names[feature_indices].tolist(),
}, ADAPTER_REPRESENTATION_JSON)
persist_artifact(ADAPTER_REPRESENTATION_JSON)

print('=' * 92)
print('[CELL 5] ADAPTER REPRESENTATION')
print('=' * 92)
print(f'Scaler fit windows            : {int(train_mask.sum())}')
print('Validation rows seen by scaler: 0')
print(f'Trial embeddings              : {trial_embeddings.shape}')
print(f'Global class prototypes       : {global_class_prototypes.shape}')
print('[PASS] Train-only adapter representation built.')


[CELL 5] ADAPTER REPRESENTATION
Scaler fit windows            : 137088
Validation rows seen by scaler: 0
Trial embeddings              : (3612, 224)
Global class prototypes       : (4, 224)
[PASS] Train-only adapter representation built.



## Bước 6 — Few-shot episode và prototype adapter

**Input:** trial table, embeddings, `k`, seed và subject.  
**Output:** calibration/evaluation split không overlap; P0/P1 score và metrics theo scope.  
**PASS:** đủ `k` calibration trials/class, evaluation còn đủ bốn classes và không có shared trial ID.


In [24]:

# === CELL 6: Leakage-safe episode builder and personalized prototype adapter ===
class_to_index = {label: index for index, label in enumerate(class_order)}
global_prob_columns = [f'global_prob__{label}' for label in class_order]


def build_episode(subject_id: str, k: int, seed: int) -> dict[str, Any]:
    subject_rows = np.flatnonzero(trial_df['subject_id'].astype(str).to_numpy() == str(subject_id))
    if len(subject_rows) == 0:
        raise RuntimeError(f'Không tìm thấy trials cho subject {subject_id}.')

    if PROFILE['calibration_pool'] == 'session1_only':
        pool_rows = subject_rows[trial_df.iloc[subject_rows]['session_id'].to_numpy(dtype=int) == 1]
    else:
        pool_rows = subject_rows

    calibration_rows: list[int] = []
    calibration_by_class: dict[str, list[str]] = {}
    for label in class_order:
        candidates = pool_rows[trial_df.iloc[pool_rows]['y_true'].astype(str).to_numpy() == label]
        if len(candidates) < k:
            raise RuntimeError(
                f'Subject={subject_id}, class={label} chỉ có {len(candidates)} calibration candidates, cần k={k}.'
            )
        rng = np.random.default_rng(stable_seed(DATASET_PROFILE, subject_id, label, k, seed))
        selected = np.sort(rng.choice(candidates, size=k, replace=False))
        calibration_rows.extend(selected.tolist())
        calibration_by_class[str(label)] = trial_df.iloc[selected]['trial_key'].astype(str).tolist()

    calibration_rows_array = np.asarray(sorted(calibration_rows), dtype=np.int64)
    evaluation_rows = np.setdiff1d(subject_rows, calibration_rows_array, assume_unique=False)

    calibration_ids = set(trial_df.iloc[calibration_rows_array]['trial_key'].astype(str))
    evaluation_ids = set(trial_df.iloc[evaluation_rows]['trial_key'].astype(str))
    if calibration_ids & evaluation_ids:
        raise RuntimeError('Calibration/evaluation trial overlap.')

    calibration_counts = trial_df.iloc[calibration_rows_array].groupby('y_true')['trial_key'].nunique().to_dict()
    if any(int(calibration_counts.get(label, 0)) != k for label in class_order):
        raise RuntimeError(f'Calibration count không đúng k={k}: {calibration_counts}')

    evaluation_labels = set(trial_df.iloc[evaluation_rows]['y_true'].astype(str))
    if evaluation_labels != set(class_order):
        raise RuntimeError(f'Evaluation không đủ classes: {evaluation_labels}')

    scopes = {'all_eval': evaluation_rows}
    if DATASET_PROFILE == 'grabmyo':
        eval_sessions = trial_df.iloc[evaluation_rows]['session_id'].to_numpy(dtype=int)
        scopes['same_session_holdout'] = evaluation_rows[eval_sessions == 1]
        scopes['cross_session'] = evaluation_rows[eval_sessions > 1]
        if len(scopes['same_session_holdout']) == 0 or len(scopes['cross_session']) == 0:
            raise RuntimeError('GRABMyo episode thiếu same-session hoặc cross-session evaluation.')

    return {
        'subject_id': str(subject_id),
        'k': int(k),
        'seed': int(seed),
        'episode_id': f'{DATASET_PROFILE}-S{subject_id}-K{k}-seed{seed}',
        'calibration_rows': calibration_rows_array,
        'evaluation_rows': evaluation_rows,
        'scopes': scopes,
        'calibration_by_class': calibration_by_class,
    }


def distance_logits(
    evaluation_embeddings: np.ndarray,
    prototypes: np.ndarray,
    metric: str,
    temperature: float,
) -> np.ndarray:
    if metric == 'sqeuclidean':
        distances = np.mean(
            (evaluation_embeddings[:, None, :] - prototypes[None, :, :]) ** 2,
            axis=2,
        )
    elif metric == 'cosine':
        eval_norm = evaluation_embeddings / np.clip(
            np.linalg.norm(evaluation_embeddings, axis=1, keepdims=True), 1e-12, None
        )
        proto_norm = prototypes / np.clip(
            np.linalg.norm(prototypes, axis=1, keepdims=True), 1e-12, None
        )
        distances = 1.0 - eval_norm @ proto_norm.T
    else:
        raise ValueError(f'Unsupported distance metric: {metric}')
    return -distances / float(temperature)


def evaluate_episode(episode: dict[str, Any], policy: dict[str, Any]) -> tuple[pd.DataFrame, list[dict[str, Any]]]:
    calibration_rows = episode['calibration_rows']
    evaluation_rows = episode['evaluation_rows']

    subject_prototypes = np.zeros_like(global_class_prototypes)
    for class_index, label in enumerate(class_order):
        class_cal_rows = calibration_rows[
            trial_df.iloc[calibration_rows]['y_true'].astype(str).to_numpy() == label
        ]
        subject_prototypes[class_index] = trial_embeddings[class_cal_rows].mean(axis=0)

    rho = float(policy['subject_prototype_weight'])
    personalized_prototypes = rho * subject_prototypes + (1.0 - rho) * global_class_prototypes

    logits = distance_logits(
        trial_embeddings[evaluation_rows],
        personalized_prototypes,
        metric=str(policy['distance_metric']),
        temperature=float(policy['temperature']),
    )
    prototype_prob = softmax_rows(logits, temperature=1.0)
    global_prob = trial_df.iloc[evaluation_rows][global_prob_columns].to_numpy(dtype=np.float64)
    alpha = float(policy['personalization_blend_weight'])
    personalized_prob = (1.0 - alpha) * global_prob + alpha * prototype_prob
    personalized_prob = personalized_prob / personalized_prob.sum(axis=1, keepdims=True)

    base = trial_df.iloc[evaluation_rows][[
        'trial_key', 'record_id', 'repetition_id', 'subject_id', 'session_id',
        'trial_ordinal', 'split_name', 'y_true', 'valid_window_count',
        'p0_pred', 'p0_correct', 'p0_confidence', 'p0_margin', 'p0_entropy',
    ]].copy().reset_index(drop=True)
    base['episode_id'] = episode['episode_id']
    base['k'] = episode['k']
    base['seed'] = episode['seed']
    base['p1_pred'] = class_order[np.argmax(personalized_prob, axis=1)]
    base['p1_correct'] = base['p1_pred'].astype(str) == base['y_true'].astype(str)
    p1_confidence, p1_margin, p1_entropy = score_diagnostics(personalized_prob)
    base['p1_confidence'] = p1_confidence
    base['p1_margin'] = p1_margin
    base['p1_entropy'] = p1_entropy
    base['distance_metric'] = policy['distance_metric']
    base['subject_prototype_weight'] = rho
    base['personalization_blend_weight'] = alpha
    base['temperature'] = float(policy['temperature'])
    base['eval_scope'] = 'all_eval'
    if DATASET_PROFILE == 'grabmyo':
        base['eval_scope'] = np.where(base['session_id'].astype(int) == 1, 'same_session_holdout', 'cross_session')

    for class_index, label in enumerate(class_order):
        base[f'p0_prob__{label}'] = global_prob[:, class_index]
        base[f'prototype_prob__{label}'] = prototype_prob[:, class_index]
        base[f'p1_prob__{label}'] = personalized_prob[:, class_index]

    metric_rows: list[dict[str, Any]] = []
    for scope_name, scope_rows in episode['scopes'].items():
        scope_trial_ids = set(trial_df.iloc[scope_rows]['trial_key'].astype(str))
        scope_frame = base[base['trial_key'].astype(str).isin(scope_trial_ids)]
        if len(scope_frame) == 0:
            continue
        p0_metrics = metric_value(scope_frame, 'y_true', 'p0_pred', class_order)
        p1_metrics = metric_value(scope_frame, 'y_true', 'p1_pred', class_order)
        metric_rows.append({
            'episode_id': episode['episode_id'],
            'subject_id': episode['subject_id'],
            'k': episode['k'],
            'seed': episode['seed'],
            'scope': scope_name,
            'n_calibration_trials': int(len(calibration_rows)),
            'n_evaluation_trials': int(len(scope_frame)),
            **{f'p0_{key}': value for key, value in p0_metrics.items()},
            **{f'p1_{key}': value for key, value in p1_metrics.items()},
            **{f'delta_{key}': p1_metrics[key] - p0_metrics[key] for key in p0_metrics},
        })
    return base, metric_rows


# One deterministic unit test on the first training subject.
_test_episode = build_episode(train_subjects[0], k=2, seed=SEEDS[0])
_test_policy = POLICY_GRID[0]
_test_predictions, _test_metrics = evaluate_episode(_test_episode, _test_policy)
assert set(trial_df.iloc[_test_episode['calibration_rows']]['trial_key']).isdisjoint(
    set(_test_predictions['trial_key'])
)
assert np.isfinite(_test_predictions[[f'p1_prob__{label}' for label in class_order]].to_numpy()).all()
print('[PASS] Episode builder and prototype adapter unit test passed.')


[PASS] Episode builder and prototype adapter unit test passed.



## Bước 7 — Chọn adaptation policy chỉ bằng training subjects

**Input:** simulated few-shot episodes trên Day 32 training subjects.  
**Output:** leaderboard và một frozen policy riêng cho `k=2` và `k=3`.  
**PASS:** validation subjects không xuất hiện trong episode cache hoặc policy selection.


In [ ]:

# === CELL 7: Train-subject episodic policy selection ===
POLICY_SEARCH_CSV = RUNTIME_ROOT / 'few-shot-policy-search.csv'
POLICY_SELECTION_JSON = RUNTIME_ROOT / 'few-shot-policy-selection.json'

selection_subjects = train_subjects[:4] if QUICK_SMOKE_TEST else train_subjects
selection_seeds = SEEDS

# Cache episode splits once; candidates see identical episodes.
train_episode_cache: dict[tuple[int, int, str], dict[str, Any]] = {}
for k in K_VALUES:
    for seed in selection_seeds:
        for subject_id in selection_subjects:
            episode = build_episode(subject_id, k, seed)
            if subject_id in validation_subjects:
                raise RuntimeError('Validation subject entered policy-selection cache.')
            train_episode_cache[(k, seed, subject_id)] = episode

policy_rows: list[dict[str, Any]] = []
selected_policies: dict[int, dict[str, Any]] = {}

for k in K_VALUES:
    print(f'[POLICY SEARCH] k={k}')
    candidate_rows: list[dict[str, Any]] = []
    for candidate_index, policy in enumerate(POLICY_GRID):
        episode_metric_rows: list[dict[str, Any]] = []
        started = time.perf_counter()
        for seed in selection_seeds:
            for subject_id in selection_subjects:
                episode = train_episode_cache[(k, seed, subject_id)]
                _, metrics = evaluate_episode(episode, policy)
                primary_rows = [row for row in metrics if row['scope'] == PROFILE['primary_selection_scope']]
                if len(primary_rows) != 1:
                    raise RuntimeError('Primary policy-selection scope missing or duplicated.')
                episode_metric_rows.append(primary_rows[0])

        episode_metrics_df = pd.DataFrame(episode_metric_rows)
        deltas = episode_metrics_df['delta_macro_f1'].to_numpy(dtype=np.float64)
        row = {
            'k': k,
            'candidate_index': candidate_index,
            'policy_id': f'k{k}-policy{candidate_index:02d}',
            **policy,
            'selection_scope': PROFILE['primary_selection_scope'],
            'episode_count': int(len(episode_metrics_df)),
            'subject_count': int(episode_metrics_df['subject_id'].nunique()),
            'seed_count': int(episode_metrics_df['seed'].nunique()),
            'p0_macro_f1_mean': float(episode_metrics_df['p0_macro_f1'].mean()),
            'p1_macro_f1_mean': float(episode_metrics_df['p1_macro_f1'].mean()),
            'delta_macro_f1_mean': float(deltas.mean()),
            'delta_macro_f1_p10': float(np.quantile(deltas, 0.10)),
            'delta_macro_f1_std': float(deltas.std(ddof=0)),
            'p1_balanced_accuracy_mean': float(episode_metrics_df['p1_balanced_accuracy'].mean()),
            'harm_episode_fraction': float(np.mean(deltas < 0)),
            'elapsed_seconds': float(time.perf_counter() - started),
        }
        candidate_rows.append(row)
        print(
            f"  {row['policy_id']} p1_macro_f1={row['p1_macro_f1_mean']:.4f} "
            f"delta={row['delta_macro_f1_mean']:+.4f} harm={row['harm_episode_fraction']:.3f}"
        )

    candidate_df = pd.DataFrame(candidate_rows).sort_values(
        [
            'p1_macro_f1_mean',
            'delta_macro_f1_mean',
            'delta_macro_f1_p10',
            'p1_balanced_accuracy_mean',
            'personalization_blend_weight',
        ],
        ascending=[False, False, False, False, True],
    ).reset_index(drop=True)
    candidate_df.insert(0, 'rank', np.arange(1, len(candidate_df) + 1))
    policy_rows.extend(candidate_df.to_dict(orient='records'))
    selected = candidate_df.iloc[0].to_dict()
    selected_policies[int(k)] = {
        'policy_id': selected['policy_id'],
        'distance_metric': selected['distance_metric'],
        'subject_prototype_weight': float(selected['subject_prototype_weight']),
        'personalization_blend_weight': float(selected['personalization_blend_weight']),
        'temperature': float(selected['temperature']),
        'selection_scope': selected['selection_scope'],
        'training_p1_macro_f1_mean': float(selected['p1_macro_f1_mean']),
        'training_delta_macro_f1_mean': float(selected['delta_macro_f1_mean']),
        'training_harm_episode_fraction': float(selected['harm_episode_fraction']),
    }

policy_search_df = pd.DataFrame(policy_rows)
write_dataframe_atomic(policy_search_df, POLICY_SEARCH_CSV)
policy_selection_payload = {
    'schema_version': 'few-shot-policy-selection.v2',
    'created_at_utc': utc_now_iso(),
    'status': 'PASS',
    'dataset_profile': DATASET_PROFILE,
    'selection_subject_partition': 'train_only',
    'selection_subject_ids': selection_subjects,
    'validation_subject_ids_used': [],
    'selection_seeds': list(selection_seeds),
    'primary_selection_scope': PROFILE['primary_selection_scope'],
    'candidate_grid_size': len(POLICY_GRID),
    'selected_policies': {str(k): value for k, value in selected_policies.items()},
    'validation_used_for_policy_selection': False,
}
write_json_atomic(policy_selection_payload, POLICY_SELECTION_JSON)
for path in (POLICY_SEARCH_CSV, POLICY_SELECTION_JSON):
    persist_artifact(path)

print('=' * 92)
print('[CELL 7] FROZEN ADAPTATION POLICIES')
print('=' * 92)
print(json.dumps(json_safe(policy_selection_payload['selected_policies']), indent=2))
print('[PASS] Policies selected using training subjects only.')


[POLICY SEARCH] k=2
  k2-policy00 p1_macro_f1=0.9917 delta=-0.0030 harm=0.116
  k2-policy01 p1_macro_f1=0.9939 delta=-0.0008 harm=0.059
  k2-policy02 p1_macro_f1=0.9849 delta=-0.0098 harm=0.246
  k2-policy03 p1_macro_f1=0.9910 delta=-0.0037 harm=0.141
  k2-policy04 p1_macro_f1=0.9500 delta=-0.0447 harm=0.660
  k2-policy05 p1_macro_f1=0.9771 delta=-0.0176 harm=0.332
  k2-policy06 p1_macro_f1=0.9927 delta=-0.0020 harm=0.096
  k2-policy07 p1_macro_f1=0.9946 delta=-0.0001 harm=0.032
  k2-policy08 p1_macro_f1=0.9868 delta=-0.0079 harm=0.221
  k2-policy09 p1_macro_f1=0.9915 delta=-0.0032 harm=0.146
  k2-policy10 p1_macro_f1=0.9483 delta=-0.0464 harm=0.653
  k2-policy11 p1_macro_f1=0.9772 delta=-0.0175 harm=0.324
  k2-policy12 p1_macro_f1=0.9930 delta=-0.0017 harm=0.096
  k2-policy13 p1_macro_f1=0.9947 delta=-0.0000 harm=0.031
  k2-policy14 p1_macro_f1=0.9859 delta=-0.0088 harm=0.229
  k2-policy15 p1_macro_f1=0.9916 delta=-0.0031 harm=0.149
  k2-policy16 p1_macro_f1=0.9388 delta=-0.0559 harm=


## Bước 8 — Frozen validation personalization experiment

**Input:** validation subjects, frozen policies và deterministic seeds.  
**Output:** calibration manifest, trial predictions và paired P0/P1 metrics.  
**PASS:** mọi episode có đúng `k × 4` calibration trials, không overlap và P0/P1 dùng cùng evaluation IDs.


In [ ]:

# === CELL 8: Run frozen validation P0 vs P1 episodes ===
validation_predictions_parts: list[pd.DataFrame] = []
validation_metric_rows: list[dict[str, Any]] = []
calibration_manifest_rows: list[dict[str, Any]] = []
leakage_overlap_count = 0

validation_run_subjects = validation_subjects[:2] if QUICK_SMOKE_TEST else validation_subjects

for k in K_VALUES:
    policy = selected_policies[int(k)]
    for seed in SEEDS:
        for subject_id in validation_run_subjects:
            episode = build_episode(subject_id, k, seed)
            calibration_ids = set(trial_df.iloc[episode['calibration_rows']]['trial_key'].astype(str))
            for row_index in episode['calibration_rows']:
                row = trial_df.iloc[int(row_index)]
                calibration_manifest_rows.append({
                    'episode_id': episode['episode_id'],
                    'subject_id': str(subject_id),
                    'k': int(k),
                    'seed': int(seed),
                    'class_label': str(row['y_true']),
                    'trial_key': str(row['trial_key']),
                    'record_id': str(row['record_id']),
                    'repetition_id': str(row['repetition_id']),
                    'session_id': int(row['session_id']),
                    'split_name': str(row['split_name']),
                })

            predictions, metrics = evaluate_episode(episode, policy)
            eval_ids = set(predictions['trial_key'].astype(str))
            overlap = calibration_ids & eval_ids
            leakage_overlap_count += len(overlap)
            if overlap:
                raise RuntimeError(f'Calibration/evaluation overlap: {sorted(overlap)}')
            predictions['selected_policy_id'] = policy['policy_id']
            validation_predictions_parts.append(predictions)
            validation_metric_rows.extend(metrics)

validation_predictions_df = pd.concat(validation_predictions_parts, ignore_index=True)
validation_metrics_df = pd.DataFrame(validation_metric_rows)
calibration_manifest_df = pd.DataFrame(calibration_manifest_rows)

# Structural gates.
calibration_counts = (
    calibration_manifest_df.groupby(['episode_id', 'class_label'])['trial_key'].nunique()
)
expected_counts = calibration_manifest_df.groupby('episode_id')['k'].first()
for episode_id, count_series in calibration_counts.groupby(level=0):
    expected_k = int(expected_counts.loc[episode_id])
    if len(count_series) != len(class_order) or not np.all(count_series.to_numpy() == expected_k):
        raise RuntimeError(f'Calibration class count gate failed: {episode_id}')

prediction_probability_columns = [
    column for column in validation_predictions_df.columns
    if column.startswith('p0_prob__') or column.startswith('p1_prob__') or column.startswith('prototype_prob__')
]
if not np.isfinite(validation_predictions_df[prediction_probability_columns].to_numpy(dtype=float)).all():
    raise RuntimeError('Validation prediction scores chứa NaN/Inf.')
if leakage_overlap_count != 0:
    raise RuntimeError(f'Leakage overlap count: {leakage_overlap_count}')

CALIBRATION_MANIFEST_CSV = RUNTIME_ROOT / 'few-shot-calibration-manifest.csv'
TRIAL_PREDICTIONS_CSV_GZ = RUNTIME_ROOT / 'few-shot-trial-predictions.csv.gz'
FEWSHOT_RESULTS_CSV = RUNTIME_ROOT / 'few-shot-results.csv'
write_dataframe_atomic(calibration_manifest_df, CALIBRATION_MANIFEST_CSV)
write_dataframe_atomic(validation_predictions_df, TRIAL_PREDICTIONS_CSV_GZ)
write_dataframe_atomic(validation_metrics_df, FEWSHOT_RESULTS_CSV)
for path in (CALIBRATION_MANIFEST_CSV, TRIAL_PREDICTIONS_CSV_GZ, FEWSHOT_RESULTS_CSV):
    persist_artifact(path)

print('=' * 92)
print('[CELL 8] VALIDATION EPISODES')
print('=' * 92)
print(f'Validation subjects           : {len(validation_run_subjects)}')
print(f'Episodes                      : {validation_metrics_df["episode_id"].nunique()}')
print(f'Calibration manifest rows     : {len(calibration_manifest_df)}')
print(f'Evaluation prediction rows    : {len(validation_predictions_df)}')
print(f'Calibration/eval overlap      : {leakage_overlap_count}')
print('[PASS] Frozen validation P0/P1 episodes completed.')



## Bước 9 — Paired analysis, subject/session/class summaries

**Input:** validation episode metrics và trial predictions.  
**Output:** paired deltas, subject-level summary, class/session summary và uncertainty at subject level.  
**PASS:** thống kê được tính trên paired evaluation denominators; bootstrap resample theo subject, không theo window.


In [ ]:

# === CELL 9: Paired summaries and subject-cluster uncertainty ===
primary_metrics_df = validation_metrics_df[
    validation_metrics_df['scope'] == PROFILE['primary_selection_scope']
].copy()

per_subject_improvement_df = (
    primary_metrics_df.groupby(['subject_id', 'k'], as_index=False)
    .agg(
        episode_count=('episode_id', 'nunique'),
        p0_macro_f1_mean=('p0_macro_f1', 'mean'),
        p1_macro_f1_mean=('p1_macro_f1', 'mean'),
        delta_macro_f1_mean=('delta_macro_f1', 'mean'),
        delta_macro_f1_std=('delta_macro_f1', lambda values: float(np.std(values, ddof=0))),
        p0_balanced_accuracy_mean=('p0_balanced_accuracy', 'mean'),
        p1_balanced_accuracy_mean=('p1_balanced_accuracy', 'mean'),
        delta_balanced_accuracy_mean=('delta_balanced_accuracy', 'mean'),
    )
)

paired_summary_rows: list[dict[str, Any]] = []
for k in K_VALUES:
    subject_rows = per_subject_improvement_df[per_subject_improvement_df['k'] == k]
    deltas = subject_rows['delta_macro_f1_mean'].to_numpy(dtype=np.float64)
    ci_low, ci_high = bootstrap_mean_ci(deltas, seed=stable_seed('bootstrap', DATASET_PROFILE, k), n_resamples=BOOTSTRAP_RESAMPLES)
    try:
        if np.allclose(deltas, 0):
            wilcoxon_statistic, wilcoxon_pvalue = 0.0, 1.0
        else:
            test = wilcoxon(deltas, alternative='two-sided', zero_method='wilcox')
            wilcoxon_statistic = float(test.statistic)
            wilcoxon_pvalue = float(test.pvalue)
    except ValueError:
        wilcoxon_statistic, wilcoxon_pvalue = float('nan'), float('nan')

    paired_summary_rows.append({
        'dataset_profile': DATASET_PROFILE,
        'k': int(k),
        'primary_scope': PROFILE['primary_selection_scope'],
        'subject_count': int(len(subject_rows)),
        'episode_count': int(primary_metrics_df[primary_metrics_df['k'] == k]['episode_id'].nunique()),
        'p0_macro_f1_subject_mean': float(subject_rows['p0_macro_f1_mean'].mean()),
        'p1_macro_f1_subject_mean': float(subject_rows['p1_macro_f1_mean'].mean()),
        'delta_macro_f1_subject_mean': float(deltas.mean()),
        'delta_macro_f1_ci95_low': ci_low,
        'delta_macro_f1_ci95_high': ci_high,
        'subjects_improved': int(np.sum(deltas > 0)),
        'subjects_equal': int(np.sum(np.isclose(deltas, 0))),
        'subjects_harmed': int(np.sum(deltas < 0)),
        'wilcoxon_statistic_exploratory': wilcoxon_statistic,
        'wilcoxon_pvalue_exploratory': wilcoxon_pvalue,
        'selected_policy_id': selected_policies[int(k)]['policy_id'],
    })

personalization_vs_global_df = pd.DataFrame(paired_summary_rows)

# Per-class paired metrics across all validation episode trial rows.
class_summary_rows: list[dict[str, Any]] = []
for k in K_VALUES:
    frame_k = validation_predictions_df[validation_predictions_df['k'] == k]
    for label in class_order:
        label_rows = frame_k[frame_k['y_true'].astype(str) == label]
        class_summary_rows.append({
            'k': int(k),
            'class_label': str(label),
            'n_evaluation_rows_across_episodes': int(len(label_rows)),
            'p0_recall': float(np.mean(label_rows['p0_correct'])),
            'p1_recall': float(np.mean(label_rows['p1_correct'])),
            'delta_recall': float(np.mean(label_rows['p1_correct']) - np.mean(label_rows['p0_correct'])),
            'p0_confidence_mean': float(label_rows['p0_confidence'].mean()),
            'p1_confidence_mean': float(label_rows['p1_confidence'].mean()),
        })
class_summary_df = pd.DataFrame(class_summary_rows)

# Session summary is especially important for GRABMyo; an empty schema is still emitted for Mendeley.
session_summary_rows: list[dict[str, Any]] = []
for (k, session_id), group in validation_predictions_df.groupby(['k', 'session_id']):
    p0_metrics = metric_value(group, 'y_true', 'p0_pred', class_order)
    p1_metrics = metric_value(group, 'y_true', 'p1_pred', class_order)
    session_summary_rows.append({
        'k': int(k),
        'session_id': int(session_id),
        'n_prediction_rows_across_episodes': int(len(group)),
        **{f'p0_{key}': value for key, value in p0_metrics.items()},
        **{f'p1_{key}': value for key, value in p1_metrics.items()},
        **{f'delta_{key}': p1_metrics[key] - p0_metrics[key] for key in p0_metrics},
    })
session_summary_df = pd.DataFrame(session_summary_rows)

PER_SUBJECT_CSV = RUNTIME_ROOT / 'per-subject-improvement.csv'
PERSONALIZATION_VS_GLOBAL_CSV = RUNTIME_ROOT / 'personalization-vs-global.csv'
CLASS_SUMMARY_CSV = RUNTIME_ROOT / 'few-shot-class-summary.csv'
SESSION_SUMMARY_CSV = RUNTIME_ROOT / 'few-shot-session-summary.csv'
write_dataframe_atomic(per_subject_improvement_df, PER_SUBJECT_CSV)
write_dataframe_atomic(personalization_vs_global_df, PERSONALIZATION_VS_GLOBAL_CSV)
write_dataframe_atomic(class_summary_df, CLASS_SUMMARY_CSV)
write_dataframe_atomic(session_summary_df, SESSION_SUMMARY_CSV)
for path in (PER_SUBJECT_CSV, PERSONALIZATION_VS_GLOBAL_CSV, CLASS_SUMMARY_CSV, SESSION_SUMMARY_CSV):
    persist_artifact(path)

print('=' * 92)
print('[CELL 9] PAIRED SUMMARY')
print('=' * 92)
display(personalization_vs_global_df)
print('[PASS] Paired subject-level summaries generated.')



## Bước 10 — Figures và Day 35-ready score artifacts

**Input:** paired subject summaries.  
**Output:** P0→P1 paired plot và distribution của subject-level delta.  
**PASS:** figures chỉ dùng frozen validation result sau khi policy đã khóa.


In [ ]:

# === CELL 10: Figures ===
PAIRED_PLOT_PNG = RUNTIME_ROOT / 'personalization-vs-global.png'
DELTA_PLOT_PNG = RUNTIME_ROOT / 'personalization-delta-distribution.png'

fig, ax = plt.subplots(figsize=(8, 6))
for k in K_VALUES:
    subset = per_subject_improvement_df[per_subject_improvement_df['k'] == k]
    x_positions = np.asarray([0, 1], dtype=float) + (k - 2) * 0.08
    for _, row in subset.iterrows():
        ax.plot(x_positions, [row['p0_macro_f1_mean'], row['p1_macro_f1_mean']], marker='o', alpha=0.45)
ax.set_xticks([0.04, 1.04], labels=['P0 global', 'P1 personalized'])
ax.set_ylabel('Subject mean macro-F1')
ax.set_title(f'Day 34 paired personalization — {DATASET_PROFILE}')
ax.grid(True, axis='y', alpha=0.25)
fig.tight_layout()
fig.savefig(PAIRED_PLOT_PNG, dpi=170, bbox_inches='tight')
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 6))
for k in K_VALUES:
    subset = per_subject_improvement_df[per_subject_improvement_df['k'] == k]
    ax.scatter(
        np.full(len(subset), k, dtype=float),
        subset['delta_macro_f1_mean'].to_numpy(dtype=float),
        label=f'k={k}',
        alpha=0.75,
    )
ax.axhline(0.0, linewidth=1.0)
ax.set_xticks(list(K_VALUES))
ax.set_xlabel('Calibration trials per class (k)')
ax.set_ylabel('P1 − P0 subject mean macro-F1')
ax.set_title(f'Personalization delta by subject — {DATASET_PROFILE}')
ax.grid(True, axis='y', alpha=0.25)
fig.tight_layout()
fig.savefig(DELTA_PLOT_PNG, dpi=170, bbox_inches='tight')
plt.close(fig)

for path in (PAIRED_PLOT_PNG, DELTA_PLOT_PNG):
    persist_artifact(path)

print(f'[OK] {PAIRED_PLOT_PNG}')
print(f'[OK] {DELTA_PLOT_PNG}')
print('[PASS] Figures generated.')



## Bước 11 — Adapter bundle, evidence, final gate và handoff ZIP

**Input:** frozen model contract, adapter scaler/prototypes, selected policies và validation evidence.  
**Output:** reusable adapter bundle, evidence JSON, final gate và ZIP bàn giao.  
**PASS:** tất cả structural/safety checks đều `True`; performance improvement không bị dùng làm gate kỹ thuật.


In [31]:
required_variables = [
    "adapter_scaler",
    "global_class_prototypes",
    "selected_policies",
    "selection_subjects",
    "validation_run_subjects",
    "validation_predictions_df",
    "validation_metrics_df",
    "calibration_manifest_df",
    "personalization_vs_global_df",
    "leakage_overlap_count",
    "p0_mismatch_count",
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    print("[NOT READY] Thiếu biến:")
    for name in missing_variables:
        print(f"  - {name}")
else:
    print("[PASS] Runtime còn đầy đủ trạng thái.")
    print("[NEXT] Chỉ cần chạy lại CELL 11.")

[PASS] Runtime còn đầy đủ trạng thái.
[NEXT] Chỉ cần chạy lại CELL 11.


In [32]:

# === CELL 11: Adapter bundle, evidence, final gate and handoff ===
FEWSHOT_PROTOCOL_JSON = RUNTIME_ROOT / 'few-shot-protocol.json'
ADAPTER_BUNDLE_JOBLIB = RUNTIME_ROOT / 'few-shot-adapter-bundle.joblib'
EVIDENCE_JSON = RUNTIME_ROOT / 'day34-personalization-evidence.json'
FINAL_GATE_JSON = RUNTIME_ROOT / 'day34-final-gate.json'
HANDOFF_ZIP = RUNTIME_ROOT / 'day34-personalization-handoff.zip'

protocol_payload = {
    'schema_version': 'few-shot-protocol.v2',
    'created_at_utc': utc_now_iso(),
    'dataset_profile': DATASET_PROFILE,
    'dataset_view_id': dataset_view_id,
    'primary_unit': 'repetition' if DATASET_PROFILE == 'mendeley' else 'trial_record',
    'class_order': class_order.tolist(),
    'k_values': list(K_VALUES),
    'episode_seeds': list(SEEDS),
    'p0': 'exact frozen Day 32 baseline; no subject calibration',
    'p1': 'train-subject-tuned shrinkage prototype adapter blended with frozen global score',
    'calibration_pool': PROFILE['calibration_pool'],
    'evaluation_policy': (
        'all non-calibration repetitions'
        if DATASET_PROFILE == 'mendeley'
        else 'session1 held-out plus all session2/session3 trials; report cross-session separately'
    ),
    'primary_selection_scope': PROFILE['primary_selection_scope'],
    'policy_selection_partition': 'train subjects only',
    'validation_policy_tuning_allowed': False,
    'baseline_refit_allowed': False,
    'test_set_opened': False,
    'clinical_use_allowed': False,
}
write_json_atomic(protocol_payload, FEWSHOT_PROTOCOL_JSON)

adapter_bundle = {
    'schema_version': 'few-shot-adapter-bundle.v2',
    'created_at_utc': utc_now_iso(),
    'dataset_profile': DATASET_PROFILE,
    'dataset_view_id': dataset_view_id,
    'npz_sha256': NPZ_SHA256,
    'baseline_model_sha256': MODEL_SHA256,
    'baseline_model_path': str(MODEL_PATH),
    'feature_indices': feature_indices,
    'feature_names': feature_names[feature_indices],
    'class_order': class_order,
    'adapter_scaler': adapter_scaler,
    'global_class_prototypes': global_class_prototypes,
    'selected_policies': selected_policies,
    'calibration_pool': PROFILE['calibration_pool'],
    'clinical_use_allowed': False,
}
temporary_bundle = ADAPTER_BUNDLE_JOBLIB.with_suffix(ADAPTER_BUNDLE_JOBLIB.suffix + '.part')
joblib.dump(adapter_bundle, temporary_bundle, compress=3)
temporary_bundle.replace(ADAPTER_BUNDLE_JOBLIB)

artifact_paths = [
    INPUT_GATE_JSON,
    P0_AUDIT_CSV,
    P0_AUDIT_JSON,
    ADAPTER_REPRESENTATION_JSON,
    POLICY_SEARCH_CSV,
    POLICY_SELECTION_JSON,
    CALIBRATION_MANIFEST_CSV,
    TRIAL_PREDICTIONS_CSV_GZ,
    FEWSHOT_RESULTS_CSV,
    PER_SUBJECT_CSV,
    PERSONALIZATION_VS_GLOBAL_CSV,
    CLASS_SUMMARY_CSV,
    SESSION_SUMMARY_CSV,
    PAIRED_PLOT_PNG,
    DELTA_PLOT_PNG,
    FEWSHOT_PROTOCOL_JSON,
    ADAPTER_BUNDLE_JOBLIB,
]
missing_artifacts = [str(path) for path in artifact_paths if not path.exists()]
if missing_artifacts:
    raise RuntimeError('Thiếu Day 34 artifacts:\n' + '\n'.join(missing_artifacts))

subject_policy_overlap = sorted(set(selection_subjects) & set(validation_run_subjects))
expected_episode_count = len(validation_run_subjects) * len(K_VALUES) * len(SEEDS)
actual_episode_count = validation_metrics_df['episode_id'].nunique()

checks = {
    'official_run_not_smoke_test': bool(not QUICK_SMOKE_TEST),
    'input_gate_passed': bool(input_gate['status'] == 'PASS'),
    'baseline_p0_reproduced_exactly': bool(p0_mismatch_count == 0),
    'baseline_model_not_refit': bool(BASELINE_REFIT_ALLOWED is False),
    'baseline_model_hash_unchanged': bool(sha256_file(MODEL_PATH) == MODEL_SHA256),
    'adapter_scaler_fit_train_only': True,
    'policy_selection_train_subjects_only': bool(len(subject_policy_overlap) == 0),
    'validation_not_used_for_policy_selection': bool(VALIDATION_POLICY_TUNING_ALLOWED is False),
    'calibration_evaluation_disjoint': bool(leakage_overlap_count == 0),
    'all_validation_episodes_completed': bool(actual_episode_count == expected_episode_count),
    'all_prediction_scores_finite': bool(
        np.isfinite(validation_predictions_df[prediction_probability_columns].to_numpy(dtype=float)).all()
    ),
    'p0_p1_paired_on_same_evaluation_rows': bool(
        validation_predictions_df[['p0_pred', 'p1_pred']].notna().all(axis=None)
    ),
    'test_set_remained_closed': bool(TEST_SET_OPENED is False),
    'pooled_dataset_blocked': bool(POOLED_DATASET_ALLOWED is False),
    'clinical_use_blocked': bool(CLINICAL_USE_ALLOWED is False),
}
failed_checks = [name for name, value in checks.items() if not isinstance(value, (bool, np.bool_)) or not bool(value)]

final_status = 'PASS' if not failed_checks else 'FAIL'
final_gate_payload = {
    'schema_version': 'day34-final-gate.v2',
    'created_at_utc': utc_now_iso(),
    'status': final_status,
    'dataset_profile': DATASET_PROFILE,
    'checks': {name: bool(value) for name, value in checks.items()},
    'counts': {
        'validation_subjects': int(len(validation_run_subjects)),
        'expected_episodes': int(expected_episode_count),
        'actual_episodes': int(actual_episode_count),
        'calibration_manifest_rows': int(len(calibration_manifest_df)),
        'evaluation_prediction_rows': int(len(validation_predictions_df)),
        'calibration_evaluation_overlap_count': int(leakage_overlap_count),
        'p0_reproduction_mismatch_count': int(p0_mismatch_count),
    },
    'failed_checks': failed_checks,
    'selected_policies': {str(k): value for k, value in selected_policies.items()},
    'performance_is_not_a_structural_gate': True,
}
write_json_atomic(final_gate_payload, FINAL_GATE_JSON)

artifact_hashes = {
    path.name: {
        'size_bytes': int(path.stat().st_size),
        'sha256': sha256_file(path),
        'runtime_path': str(path),
        'persistent_path': str(PERSIST_ROOT / path.name),
    }
    for path in artifact_paths + [FINAL_GATE_JSON]
}

evidence_payload = {
    'schema_version': 'day34-personalization-evidence.v2',
    'created_at_utc': utc_now_iso(),
    'status': final_status,
    'run_id': RUN_ID,
    'scope': 'personalization_few_shot_adaptation',
    'input': input_gate,
    'protocol': protocol_payload,
    'policy_selection': policy_selection_payload,
    'validation_primary_summary': personalization_vs_global_df.to_dict(orient='records'),
    'governance': {
        'baseline_refit_allowed': BASELINE_REFIT_ALLOWED,
        'validation_policy_tuning_allowed': VALIDATION_POLICY_TUNING_ALLOWED,
        'test_set_opened': TEST_SET_OPENED,
        'pooled_dataset_allowed': POOLED_DATASET_ALLOWED,
        'clinical_use_allowed': CLINICAL_USE_ALLOWED,
    },
    'technical_debt': [
        'Prototype adapter is evaluated offline at trial/repetition level.',
        'Confidence calibration and abstention are deferred to Day 35.',
        'Performance deltas must be interpreted with subject-cluster uncertainty, not window count.',
    ],
    'artifacts': artifact_hashes,
    'next_step': 'Day 35 confidence calibration and abstention using P0/P1 probability, margin and entropy columns.',
}
write_json_atomic(evidence_payload, EVIDENCE_JSON)

# Persist final artifacts after evidence/gate creation.
for path in (FEWSHOT_PROTOCOL_JSON, ADAPTER_BUNDLE_JOBLIB, EVIDENCE_JSON, FINAL_GATE_JSON):
    persist_artifact(path)

# Handoff ZIP contains only final reproducible artifacts, not raw NPZ or baseline binary.
zip_members = artifact_paths + [EVIDENCE_JSON, FINAL_GATE_JSON]
temporary_zip = HANDOFF_ZIP.with_suffix(HANDOFF_ZIP.suffix + '.part')
temporary_zip.unlink(missing_ok=True)
with zipfile.ZipFile(temporary_zip, mode='w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in zip_members:
        archive.write(path, arcname=path.name)
temporary_zip.replace(HANDOFF_ZIP)
persist_artifact(HANDOFF_ZIP)

if failed_checks:
    raise RuntimeError(
        'Day 34 final gate failed: ' + json.dumps(failed_checks, ensure_ascii=False)
    )

print('=' * 92)
print('[CELL 11] DAY 34 FINAL SUMMARY')
print('=' * 92)
print(f'Status                        : {final_status}')
print(f'Dataset profile               : {DATASET_PROFILE}')
print(f'Validation subjects           : {len(validation_run_subjects)}')
print(f'Validation episodes           : {actual_episode_count}')
print(f'P0 reproduction mismatches    : {p0_mismatch_count}')
print(f'Calibration/eval overlaps     : {leakage_overlap_count}')
print(f'Adapter bundle                : {PERSIST_ROOT / ADAPTER_BUNDLE_JOBLIB.name}')
print(f'Evidence                      : {PERSIST_ROOT / EVIDENCE_JSON.name}')
print(f'Final gate                    : {PERSIST_ROOT / FINAL_GATE_JSON.name}')
print(f'Handoff ZIP                   : {PERSIST_ROOT / HANDOFF_ZIP.name}')
print('[PASS] Day 34 personalization/few-shot adaptation completed.')

if DOWNLOAD_HANDOFF_TO_BROWSER and IN_COLAB:
    files.download(str(PERSIST_ROOT / HANDOFF_ZIP.name))


[CELL 11] DAY 34 FINAL SUMMARY
Status                        : PASS
Dataset profile               : grabmyo
Validation subjects           : 9
Validation episodes           : 360
P0 reproduction mismatches    : 0
Calibration/eval overlaps     : 0
Adapter bundle                : /content/drive/MyDrive/MyoLab-AI-data/grabmyo-physionet-v1.1.0/outputs/day34-personalization/day34-personalization-grabmyo-v2/few-shot-adapter-bundle.joblib
Evidence                      : /content/drive/MyDrive/MyoLab-AI-data/grabmyo-physionet-v1.1.0/outputs/day34-personalization/day34-personalization-grabmyo-v2/day34-personalization-evidence.json
Final gate                    : /content/drive/MyDrive/MyoLab-AI-data/grabmyo-physionet-v1.1.0/outputs/day34-personalization/day34-personalization-grabmyo-v2/day34-final-gate.json
Handoff ZIP                   : /content/drive/MyDrive/MyoLab-AI-data/grabmyo-physionet-v1.1.0/outputs/day34-personalization/day34-personalization-grabmyo-v2/day34-personalization-handoff.zip


## Artifacts cần giữ sau Day 34

File bàn giao chính:

```text
day34-personalization-handoff.zip
```

Các artifact quan trọng cho Day 35:

```text
few-shot-trial-predictions.csv.gz
few-shot-results.csv
per-subject-improvement.csv
personalization-vs-global.csv
few-shot-policy-selection.json
few-shot-adapter-bundle.joblib
day34-personalization-evidence.json
day34-final-gate.json
```

`few-shot-trial-predictions.csv.gz` đã chứa probability vector, confidence, margin và entropy cho cả P0/P1 để tiếp tục confidence calibration và abstention.
